# CodeRAG-Bench (NAACL Findings 2025) — full basic + open-domain

**Paper:** [CodeRAG-Bench: Can Retrieval Augment Code Generation?](https://arxiv.org/abs/2406.14497) — Wang et al.  
**Site:** [code-rag-bench.github.io](https://code-rag-bench.github.io/) · **HF:** [`code-rag-bench`](https://huggingface.co/code-rag-bench)

## Indexed dataset (this notebook) — NOT a toy subset

| | |
|--|--|
| **Questions** | HumanEval **164** + MBPP **500** + DS-1000 **1,000** + ODEX **439** = **~2,103** |
| **Retrieval corpus** | programming-solutions (leave-gold-out) + **library-documentation (~34k)** |
| **On disk** | `data/corpus_code_rag/` · `data/qa/code_rag_eval.json` |
| **Results** | `results_code_rag/` |

Optional StackOverflow (~76k posts): `python scripts/run_code_rag_benchmark.py ... --stackoverflow`

Canonical HumanEval/MBPP solutions are **excluded** from the datastore (CodeRAG-Bench protocol).

Token / type slices on this full catalog: `notebooks/code_rag_tokens.ipynb`. Do not read a 20-Q leftover as the paper result.



In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")

from rag_benchmark import build_code_rag_bench_subset
from rag_benchmark.charts import METHOD_LABELS
from rag_benchmark.metric_autopsy import (
    enrich_accuracy,
    method_metric_profile,
    question_catalog,
    scenario_dual_leaderboard,
    write_autopsy_artifacts,
)

RESULTS = ROOT / "results_code_rag"
RUN_BENCHMARK = False  # True to call scripts/run_code_rag_benchmark.py from a shell

built = build_code_rag_bench_subset(project_root=ROOT)
display(Markdown("### Subset meta"))
display(pd.Series(built["meta"]).to_frame("value"))
print(f"Corpus docs: {len(list(Path(built['corpus_dir']).glob('*.txt')))}")

### Subset meta

,value
dataset,CodeRAG-Bench
paper,"Wang et al., NAACL Findings 2025"
citation,https://arxiv.org/abs/2406.14497
homepage,https://code-rag-bench.github.io/
slice,basic_programming
source,code-rag-bench/humaneval + mbpp + programming-...
n_questions,664
n_documents,464
type_counts,"{'humaneval': 164, 'mbpp': 500}"
corpus_dir,/Users/jimmyscray/Code/rag-benchmark/data/corp...


Corpus docs: 464


## How the data is structured

```
HF code-rag-bench/humaneval  (164)     HF code-rag-bench/mbpp (500)
        │                                       │
        └──────────► eval questions ────────────┘
                         │
HF code-rag-bench/programming-solutions (1128)
                         │  drop gold task_ids
                         ▼
              data/corpus_code_rag/*.txt
                         │
                         ▼
              Semantic / rerank / BM25+dense / Frontier indexes
```

| Source field | Becomes |
|--------------|---------|
| HumanEval `prompt` + `canonical_solution` | question + expected answer |
| MBPP `text` + `code` | question + expected answer |
| programming-solutions `text` / `title` / `meta.task_id` | `# title\nTask: id\n\ncode` files |


In [2]:
qa = json.loads(Path(built["qa_path"]).read_text())
display(Markdown(f"### Question catalog preview (n={len(qa)})"))
display(pd.DataFrame(qa)[["id", "code_rag_type", "question"]].assign(
    question=lambda d: d.question.str.slice(0, 120)
).groupby("code_rag_type").head(3))

sample = sorted(Path(built["corpus_dir"]).glob("*.txt"))[0]
display(Markdown(f"### Sample retrieval doc `{sample.name}`"))
print(sample.read_text(encoding="utf-8")[:700])

### Question catalog preview (n=664)

,id,code_rag_type,question
0,HumanEval-0,humaneval,Complete the following Python function. Return...
1,HumanEval-1,humaneval,Complete the following Python function. Return...
2,HumanEval-2,humaneval,Complete the following Python function. Return...
164,MBPP-11,mbpp,Write a Python function for the following prob...
165,MBPP-12,mbpp,Write a Python function for the following prob...
166,MBPP-13,mbpp,Write a Python function for the following prob...


### Sample retrieval doc `0000_601.txt`

# 601
Task: 601

# Write a function to find the longest chain which can be formed from the given set of pairs.
class Pair(object): 
	def __init__(self, a, b): 
		self.a = a 
		self.b = b 
def max_chain_length(arr, n): 
	max = 0
	mcl = [1 for i in range(n)] 
	for i in range(1, n): 
		for j in range(0, i): 
			if (arr[i].a > arr[j].b and
				mcl[i] < mcl[j] + 1): 
				mcl[i] = mcl[j] + 1
	for i in range(n): 
		if (max < mcl[i]): 
			max = mcl[i] 
	return max



## Run the benchmark

From the project root (full 664 questions — vector methods only; GraphRAG optional):

```bash
PYTHONPATH=src python scripts/run_code_rag_benchmark.py
# smoke:
PYTHONPATH=src python scripts/run_code_rag_benchmark.py semantic_rag,rerank_semantic 40
```

Then re-run the cells below to load `results_code_rag/`.


In [3]:
acc_path = RESULTS / "accuracy_results.csv"
summary_path = RESULTS / "summary.csv"
if not acc_path.exists():
    display(Markdown(
        "_No `results_code_rag/accuracy_results.csv` yet. "
        "Run `python scripts/run_code_rag_benchmark.py` first._"
    ))
else:
    write_autopsy_artifacts(
        results_dir=RESULTS,
        qa_path=Path(built["qa_path"]),
        type_key="code_rag_type",
        scenario_col="code_rag_type",
    )
    acc = enrich_accuracy(pd.read_csv(acc_path))
    qa_by = {q["id"]: q for q in json.loads(Path(built["qa_path"]).read_text())}
    acc["code_rag_type"] = acc["question_id"].map(lambda i: qa_by.get(i, {}).get("code_rag_type"))
    display(Markdown("### Summary"))
    display(pd.read_csv(summary_path))
    display(Markdown("### Dual scoreboard by task type"))
    display(scenario_dual_leaderboard(acc, scenario_col="code_rag_type"))
    display(Markdown("### Method profile"))
    display(method_metric_profile(acc)[
        ["label", "generative", "extractive", "composite", "llm_judge", "contains"]
    ].round(3))
    display(Markdown("### Question catalog w/ scores"))
    cat = question_catalog(acc, Path(built["qa_path"]), type_key="code_rag_type")
    display(cat[["label", "type", "question", "gold", "avg_judge", "avg_f1", "em_rate"]].head(12))


### Summary

,method,mean_composite_score,mean_generative_score,mean_extractive_score,mean_llm_judge,mean_token_f1,exact_match_rate,contains_answer_rate,total_tokens,prompt_tokens,completion_tokens,estimated_cost_usd,index_seconds,mean_query_latency_seconds,p95_query_latency_seconds,total_elapsed_seconds,tokens_per_query
0,semantic_rag,0.270239,0.4025,0.137978,0.505,0.275956,0.0,0.30,138091,135768,2323,0.0,3.535680,2.474349,5.034682,80.897112,6904.55
1,rerank_semantic,0.302401,0.4550,0.149802,0.460,0.299604,0.0,0.45,136351,133761,2590,0.0,3.169251,5.746950,15.878483,188.894715,6817.55


### Dual scoreboard by task type

,scenario,generative_winner,generative_score,extractive_winner,extractive_score,composite_winner,composite_score,ranking_flips
0,humaneval,Vector + rerank,0.535,Vector + rerank,0.149,Vector + rerank,0.342,False
1,mbpp,Vector + rerank,0.375,Vector + rerank,0.151,Vector + rerank,0.263,False


### Method profile

,label,generative,extractive,composite,llm_judge,contains
0,Vector + rerank,0.455,0.150,0.302,0.460,0.45
1,Semantic (vector),0.402,0.138,0.270,0.505,0.30


### Question catalog w/ scores

,label,type,question,gold,avg_judge,avg_f1,em_rate
0,Q1,humaneval,Complete the following Python function. Return...,from typing import List\n\n\ndef has_close_ele...,0.40,0.219512,0.0
1,Q2,humaneval,Complete the following Python function. Return...,from typing import List\n\n\ndef separate_pare...,0.55,0.259259,0.0
2,Q3,humaneval,Complete the following Python function. Return...,def truncate_number(number: float) -> float:\n...,0.25,0.296296,0.0
3,Q4,humaneval,Complete the following Python function. Return...,from typing import List\n\n\ndef below_zero(op...,0.50,0.194444,0.0
4,Q5,humaneval,Complete the following Python function. Return...,from typing import List\n\n\ndef mean_absolute...,0.25,0.298496,0.0
5,Q6,humaneval,Complete the following Python function. Return...,from typing import List\n\n\ndef intersperse(n...,0.50,0.428571,0.0
6,Q7,humaneval,Complete the following Python function. Return...,from typing import List\n\n\ndef parse_nested_...,0.25,0.367347,0.0
7,Q8,humaneval,Complete the following Python function. Return...,from typing import List\n\n\ndef filter_by_sub...,0.25,0.305556,0.0
8,Q9,humaneval,Complete the following Python function. Return...,"from typing import List, Tuple\n\n\ndef sum_pr...",0.80,0.413793,0.0
9,Q10,humaneval,Complete the following Python function. Return...,"from typing import List, Tuple\n\n\ndef rollin...",0.00,0.122159,0.0


## Why this dataset (vs Hotpot)

| | HotpotQA | CodeRAG-Bench (this notebook) |
|--|----------|-------------------------------|
| Domain | Wikipedia multi-hop QA | **Code generation + retrieval** |
| Year focus | 2018 classic | **2024/2025** |
| Gold | short span | **full function / program** |
| Corpus | distractor wiki paragraphs | **other programming solutions** |
| Protocol | EM-friendly | leave-gold-out retrieval |

Use Hotpot / MultiHop / GraphRAG-Bench for *when graphs help on text*.  
Use CodeRAG-Bench when the question is *does RAG help coding models?*
